# **FOREWORD**

This is adapted from 2 public kernels -
1. [LB 0.872](https://www.kaggle.com/code/tonylica/birdclef-lb-0-872-0-862-16mins-runtime)
2. [LB 0.862](https://www.kaggle.com/code/aidensong123/birdclef-2026-sed-baseline-lb-0-862)

I am using OpenVINO to infer these models and am blending them in a separate script using a simple addition. 

**Why I do this** <br>
1. This will help me add/ remove scripts at will
2. It offers better readibility
3. I can execute these scripts using a single main cell

Some of my other work in this competition are as below- <br>
1. https://www.kaggle.com/code/ravi20076/birdclef2026-public-blend-v1
2. https://www.kaggle.com/code/ravi20076/birdclef2026-public-blend-v2
3. https://www.kaggle.com/code/ravi20076/birdclef2026-public-blend-v3
4. https://www.kaggle.com/code/ravi20076/birdclef2026-public-blend-v4
5. https://www.kaggle.com/code/ravi20076/birdclef2026-public-blend-v5
6. https://www.kaggle.com/code/ravi20076/birdclef2026-openvino-starter-v1
7. https://www.kaggle.com/code/ravi20076/birdclef2026-supplements-v1
8. https://www.kaggle.com/code/ravi20076/birdclef2026-preprocessing-v1

# **SCRIPTS**

## **LB 0.872**

In [ ]:
%%writefile lb872.py

# ============================================================
# BirdCLEF 2026 | Model 1 Inference
# MODEL: LB872.pt  (finetuned, EfficientNet-B0 SED, val_auc ~0.996)
# OUTPUT: lb872.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "lb872.csv"

# ── CONFIG ────────────────────────────────────────────────────
@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    max_workers: int = 4

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)

cfg = Config()

# ── PATHS ─────────────────────────────────────────────────────
DATA_ROOT    = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR     = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
CKPT         = "/kaggle/input/datasets/tonylica/birdclef-2026-model/LB872.pt"

assert os.path.exists(CKPT), f"Missing checkpoint: {CKPT}"

# ── MODEL ─────────────────────────────────────────────────────
class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        return {
            "clipwise_logit": clipwise_logit,
            "clipwise_prob":  torch.sigmoid(clipwise_logit),
        }

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=False, in_chans=cfg.in_channels,
            features_only=False, global_pool="", num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head     = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))

class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.mel(waveforms), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.db(mel), nan=-80.0, posinf=0.0, neginf=-80.0)
        B = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mel_min  = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max  = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)

# ── LOAD MODEL ────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def safe_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

sample_sub    = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES       = list(sample_sub.columns[1:])
cfg.num_classes = len(SPECIES)

ckpt = safe_load(CKPT)
state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
epoch = ckpt.get("epoch", "?") if isinstance(ckpt, dict) else "?"
auc   = ckpt.get("metrics", {}).get("macro_auc", "?") if isinstance(ckpt, dict) else "?"

model = SEDModel(cfg)
model.load_state_dict(state, strict=True)
model.to(device).eval()
print(f"LB872 (finetuned): loaded | epoch={epoch} | val_auc={auc}")

mel_transform = MelSpectrogramTransform(cfg).to(device).eval()

# ── FILE DISCOVERY ────────────────────────────────────────────
row_pattern = re.compile(r"^(.*)_(\d+)$")

def parse_row_id(rid):
    m = row_pattern.match(str(rid))
    return (m.group(1), int(m.group(2))) if m else (None, None)

expected_ids    = sample_sub["row_id"].tolist()
expected_by_stem = {}
for rid in expected_ids:
    stem, end_sec = parse_row_id(rid)
    if stem:
        expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.ogg")))
if not test_files:
    for stem in sorted(expected_by_stem):
        p = os.path.join(TRAIN_SC_DIR, f"{stem}.ogg")
        if os.path.exists(p):
            test_files.append(p)
    print(f"No test .ogg found; using {len(test_files)} train_soundscapes fallback files.")
else:
    print(f"Found {len(test_files)} test soundscape files.")

# ── AUDIO LOADING ─────────────────────────────────────────────
def load_soundscape(path):
    y, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32), Path(path).stem

print("Loading audio...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
    results = list(ex.map(load_soundscape, test_files))
print(f"Loaded {len(results)} files in {time.time()-t0:.1f}s")

# ── INFERENCE ─────────────────────────────────────────────────
CHUNK = cfg.chunk_samples
all_row_ids, all_preds = [], []

print("Running inference...")
t0 = time.time()
with torch.no_grad():
    for audio, stem in results:
        if stem in expected_by_stem:
            n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration)
        else:
            n_chunks = max(1, len(audio) // CHUNK)

        padded_len = n_chunks * CHUNK
        audio = np.pad(audio, (0, max(0, padded_len - len(audio))))[:padded_len]

        # TRICK 3: Do NOT peak-normalize raw audio — prevents quiet files from being amplified into static
        # peak = np.abs(audio).max()
        # if peak > 0:
        #     audio = audio / peak

        chunks_tensor = torch.from_numpy(audio.reshape(n_chunks, CHUNK)).float().to(device)
        mel  = mel_transform(chunks_tensor)
        out  = model(mel)

        probs = torch.nan_to_num(out["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0)
        probs = probs.clamp(0.0, 1.0).cpu().numpy()

        # TRICK 2: Confidence-Sharpened Smoothing
        if n_chunks > 4:
            SHARPEN_POWER  = 1.5
            probs_sharp    = probs ** SHARPEN_POWER
            SMOOTH_WEIGHTS = np.array([0.05, 0.15, 0.60, 0.15, 0.05])
            p_pad = np.pad(probs_sharp, ((2, 2), (0, 0)), mode='edge')
            smoothed = (SMOOTH_WEIGHTS[0] * p_pad[:-4] +
                        SMOOTH_WEIGHTS[1] * p_pad[1:-3] +
                        SMOOTH_WEIGHTS[2] * p_pad[2:-2] +
                        SMOOTH_WEIGHTS[3] * p_pad[3:-1] +
                        SMOOTH_WEIGHTS[4] * p_pad[4:])
            probs = smoothed ** (1.0 / SHARPEN_POWER)
        elif n_chunks > 2:
            SMOOTH_WEIGHTS = np.array([0.20, 0.60, 0.20])
            p_pad = np.pad(probs, ((1, 1), (0, 0)), mode='edge')
            probs = (SMOOTH_WEIGHTS[0] * p_pad[:-2] +
                     SMOOTH_WEIGHTS[1] * p_pad[1:-1] +
                     SMOOTH_WEIGHTS[2] * p_pad[2:])

        # TRICK 1: Global Soundscape Prior (File-Max Leakage)
        FILE_MAX_WEIGHT = 0.05
        file_max = np.max(probs, axis=0, keepdims=True)
        probs = probs + (FILE_MAX_WEIGHT * file_max)
        # Intentionally not clipping — AUC handles values > 1.0 fine; clipping creates ties

        if stem in expected_by_stem:
            valid = [(e // int(cfg.chunk_duration)) - 1 for e in expected_by_stem[stem]]
            valid = [i for i in valid if 0 <= i < n_chunks]
        else:
            valid = list(range(n_chunks))

        all_row_ids.extend([f"{stem}_{(i+1)*int(cfg.chunk_duration)}" for i in valid])
        all_preds.extend(probs[valid])

print(f"Inference done in {time.time()-t0:.1f}s")

# ── SAVE ──────────────────────────────────────────────────────
expected_ids = sample_sub["row_id"].tolist()

if len(all_preds) == 0:
    pred_df = pd.DataFrame(
        np.zeros((0, len(SPECIES)), dtype=np.float32),
        columns=SPECIES,
        index=pd.Index([], name="row_id"),
    )
else:
    pred_df = pd.DataFrame(
        np.asarray(all_preds, dtype=np.float32),
        columns=SPECIES,
        index=pd.Index(all_row_ids, name="row_id"),
    )

pred_df = pred_df[~pred_df.index.duplicated(keep="first")]

missing = [rid for rid in expected_ids if rid not in pred_df.index]
if missing:
    pred_df = pd.concat([pred_df, pd.DataFrame(
        np.zeros((len(missing), len(SPECIES)), dtype=np.float32),
        columns=SPECIES, index=pd.Index(missing, name="row_id"),
    )])

pred_df = pred_df.loc[expected_ids]
pred_df.to_csv(SUBMISSION_FILE, index=True)
print(f"Saved {SUBMISSION_FILE} - shape = {pred_df.shape}\n\n")

## **LB 0.862**

In [ ]:
%%writefile lb862.py

# ============================================================
# BirdCLEF 2026 | Model 2 Inference
# MODEL: LB862.pt  (baseline, EfficientNet-B0 SED, val_auc ~0.948)
# OUTPUT: lb862.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "lb862.csv"

# ── CONFIG ────────────────────────────────────────────────────
@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    max_workers: int = 4

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)

cfg = Config()

# ── PATHS ─────────────────────────────────────────────────────
DATA_ROOT    = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR     = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
CKPT         = "/kaggle/input/datasets/tonylica/birdclef-2026-model/LB862.pt"

assert os.path.exists(CKPT), f"Missing checkpoint: {CKPT}"

# ── MODEL ─────────────────────────────────────────────────────
class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        return {
            "clipwise_logit": clipwise_logit,
            "clipwise_prob":  torch.sigmoid(clipwise_logit),
        }

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=False, in_chans=cfg.in_channels,
            features_only=False, global_pool="", num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head     = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))

class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.mel(waveforms), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.db(mel), nan=-80.0, posinf=0.0, neginf=-80.0)
        B = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mel_min  = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max  = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)

# ── LOAD MODEL ────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def safe_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

sample_sub      = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES         = list(sample_sub.columns[1:])
cfg.num_classes = len(SPECIES)

ckpt  = safe_load(CKPT)
state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
epoch = ckpt.get("epoch", "?") if isinstance(ckpt, dict) else "?"
auc   = ckpt.get("metrics", {}).get("macro_auc", "?") if isinstance(ckpt, dict) else "?"

model = SEDModel(cfg)
model.load_state_dict(state, strict=True)
model.to(device).eval()
print(f"LB862 (baseline): loaded | epoch={epoch} | val_auc={auc}")

mel_transform = MelSpectrogramTransform(cfg).to(device).eval()

# ── FILE DISCOVERY ────────────────────────────────────────────
row_pattern = re.compile(r"^(.*)_(\d+)$")

def parse_row_id(rid):
    m = row_pattern.match(str(rid))
    return (m.group(1), int(m.group(2))) if m else (None, None)

expected_ids     = sample_sub["row_id"].tolist()
expected_by_stem = {}
for rid in expected_ids:
    stem, end_sec = parse_row_id(rid)
    if stem:
        expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.ogg")))
if not test_files:
    for stem in sorted(expected_by_stem):
        p = os.path.join(TRAIN_SC_DIR, f"{stem}.ogg")
        if os.path.exists(p):
            test_files.append(p)
    print(f"No test .ogg found; using {len(test_files)} train_soundscapes fallback files.")
else:
    print(f"Found {len(test_files)} test soundscape files.")

# ── AUDIO LOADING ─────────────────────────────────────────────
def load_soundscape(path):
    y, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32), Path(path).stem

print("Loading audio...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
    results = list(ex.map(load_soundscape, test_files))
print(f"Loaded {len(results)} files in {time.time()-t0:.1f}s")

# ── INFERENCE ─────────────────────────────────────────────────
CHUNK = cfg.chunk_samples
all_row_ids, all_preds = [], []

print("Running inference...")
t0 = time.time()
with torch.no_grad():
    for audio, stem in results:
        if stem in expected_by_stem:
            n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration)
        else:
            n_chunks = max(1, len(audio) // CHUNK)

        padded_len = n_chunks * CHUNK
        audio = np.pad(audio, (0, max(0, padded_len - len(audio))))[:padded_len]

        # TRICK 3: Do NOT peak-normalize raw audio — prevents quiet files from being amplified into static
        # peak = np.abs(audio).max()
        # if peak > 0:
        #     audio = audio / peak

        chunks_tensor = torch.from_numpy(audio.reshape(n_chunks, CHUNK)).float().to(device)
        mel  = mel_transform(chunks_tensor)
        out  = model(mel)

        probs = torch.nan_to_num(out["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0)
        probs = probs.clamp(0.0, 1.0).cpu().numpy()

        # TRICK 2: Confidence-Sharpened Smoothing
        if n_chunks > 4:
            SHARPEN_POWER  = 1.5
            probs_sharp    = probs ** SHARPEN_POWER
            SMOOTH_WEIGHTS = np.array([0.05, 0.15, 0.60, 0.15, 0.05])
            p_pad = np.pad(probs_sharp, ((2, 2), (0, 0)), mode='edge')
            smoothed = (SMOOTH_WEIGHTS[0] * p_pad[:-4] +
                        SMOOTH_WEIGHTS[1] * p_pad[1:-3] +
                        SMOOTH_WEIGHTS[2] * p_pad[2:-2] +
                        SMOOTH_WEIGHTS[3] * p_pad[3:-1] +
                        SMOOTH_WEIGHTS[4] * p_pad[4:])
            probs = smoothed ** (1.0 / SHARPEN_POWER)
        elif n_chunks > 2:
            SMOOTH_WEIGHTS = np.array([0.20, 0.60, 0.20])
            p_pad = np.pad(probs, ((1, 1), (0, 0)), mode='edge')
            probs = (SMOOTH_WEIGHTS[0] * p_pad[:-2] +
                     SMOOTH_WEIGHTS[1] * p_pad[1:-1] +
                     SMOOTH_WEIGHTS[2] * p_pad[2:])

        # TRICK 1: Global Soundscape Prior (File-Max Leakage)
        FILE_MAX_WEIGHT = 0.05
        file_max = np.max(probs, axis=0, keepdims=True)
        probs = probs + (FILE_MAX_WEIGHT * file_max)
        # Intentionally not clipping — AUC handles values > 1.0 fine; clipping creates ties

        if stem in expected_by_stem:
            valid = [(e // int(cfg.chunk_duration)) - 1 for e in expected_by_stem[stem]]
            valid = [i for i in valid if 0 <= i < n_chunks]
        else:
            valid = list(range(n_chunks))

        all_row_ids.extend([f"{stem}_{(i+1)*int(cfg.chunk_duration)}" for i in valid])
        all_preds.extend(probs[valid])

print(f"Inference done in {time.time()-t0:.1f}s")

# ── SAVE ──────────────────────────────────────────────────────
expected_ids = sample_sub["row_id"].tolist()

if len(all_preds) == 0:
    pred_df = pd.DataFrame(
        np.zeros((0, len(SPECIES)), dtype=np.float32),
        columns=SPECIES,
        index=pd.Index([], name="row_id"),
    )
else:
    pred_df = pd.DataFrame(
        np.asarray(all_preds, dtype=np.float32),
        columns=SPECIES,
        index=pd.Index(all_row_ids, name="row_id"),
    )

pred_df = pred_df[~pred_df.index.duplicated(keep="first")]

missing = [rid for rid in expected_ids if rid not in pred_df.index]
if missing:
    pred_df = pd.concat([pred_df, pd.DataFrame(
        np.zeros((len(missing), len(SPECIES)), dtype=np.float32),
        columns=SPECIES, index=pd.Index(missing, name="row_id"),
    )])

pred_df = pred_df.loc[expected_ids]
pred_df.to_csv(SUBMISSION_FILE, index=True)
print(f"Saved {SUBMISSION_FILE} - shape = {pred_df.shape}\n\n")

## **BLEND**

In [ ]:
%%writefile blend.py

# ============================================================
# BirdCLEF 2026 | Blend Script
# Reads lb872.csv (finetuned) and lb862.csv (baseline), writes submission.csv
# ============================================================

import os
import argparse
import numpy as np
import pandas as pd
from scipy.special import expit as sigmoid

parser = argparse.ArgumentParser()
parser.add_argument("--exp_id", type=int, default=2, help="Blend strategy")
args   = parser.parse_args()
EXP_ID = args.exp_id

BLEND_OPTIONS = [
    ("finetuned_only",    {"mode": "prob",  "ft": 1.0,  "base": 0.0}),
    ("baseline_only",     {"mode": "prob",  "ft": 0.0,  "base": 1.0}),
    ("prob_ft80_base20",  {"mode": "prob",  "ft": 0.8,  "base": 0.2}),
    ("prob_ft70_base30",  {"mode": "prob",  "ft": 0.7,  "base": 0.3}),
    ("prob_ft50_base50",  {"mode": "prob",  "ft": 0.5,  "base": 0.5}),
    ("logit_ft80_base20", {"mode": "logit", "ft": 0.8,  "base": 0.2}),
    ("logit_ft70_base30", {"mode": "logit", "ft": 0.7,  "base": 0.3}),
    ("logit_ft50_base50", {"mode": "logit", "ft": 0.5,  "base": 0.5}),
    ("prob_ft84_base16",  {"mode": "prob",  "ft": 0.84, "base": 0.16}),
    ("prob_ft88_base12",  {"mode": "prob",  "ft": 0.88, "base": 0.12}),
    ("prob_ft75_base25",  {"mode": "prob",  "ft": 0.75, "base": 0.25}),
]

assert 0 <= EXP_ID < len(BLEND_OPTIONS), f"--exp_id must be in [0, {len(BLEND_OPTIONS)-1}]"
SELECTED_NAME, SPEC = BLEND_OPTIONS[EXP_ID]
print(f"Blend: EXP_ID={EXP_ID} -> {SELECTED_NAME}")

# ── LOAD PREDICTIONS ──────────────────────────────────────────
DATA_ROOT    = "/kaggle/input/competitions/birdclef-2026"
sample_sub   = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES      = list(sample_sub.columns[1:])
expected_ids = sample_sub["row_id"].tolist()

df_ft   = pd.read_csv("lb872.csv", index_col="row_id")[SPECIES]
df_base = pd.read_csv("lb862.csv", index_col="row_id")[SPECIES]

assert df_ft.index.tolist() == df_base.index.tolist(), \
    "row_id mismatch between model outputs — run both model scripts on the same test set."

preds_ft   = df_ft.values
preds_base = df_base.values

# ── BLEND ─────────────────────────────────────────────────────
if SPEC["mode"] == "prob":
    blended = SPEC["ft"] * preds_ft + SPEC["base"] * preds_base
elif SPEC["mode"] == "logit":
    eps = 1e-7
    logits_ft   = np.log(np.clip(preds_ft,   eps, 1-eps) / (1 - np.clip(preds_ft,   eps, 1-eps)))
    logits_base = np.log(np.clip(preds_base, eps, 1-eps) / (1 - np.clip(preds_base, eps, 1-eps)))
    blended = sigmoid(SPEC["ft"] * logits_ft + SPEC["base"] * logits_base)
else:
    raise ValueError(f"Unknown blend mode: {SPEC['mode']}")

blended = np.nan_to_num(
    blended, nan=0.0, posinf=1.0, neginf=0.0
).clip(0.0, 1.0).astype(np.float32)

# ── BUILD SUBMISSION ──────────────────────────────────────────
submission = pd.DataFrame(blended, columns=SPECIES, index=df_ft.index)
submission = submission.loc[expected_ids].reset_index()

submission.to_csv("submission.csv", index=False)
print(f"Saved: submission.csv  |  {submission.shape}")

# **SUBMISSION**

In [ ]:
!python lb872.py
!python lb862.py
!python blend.py --exp_id 10

import pandas as pd
print()
display(
    pd.read_csv(f"submission.csv")
    .head(5)
    .style
    .set_caption(f"Submission file")
)
!ls